# Numerical Comparison of Restart Methods of the Arnoldi Method

## Imports

In [ ]:
using LinearAlgebra
using JacobiDavidson
using LinearMaps
using MatrixDepot
ENV["GKSwstype"] = "nul"
using Plots
using Hungarian
using ProgressMeter
using LaTeXStrings

include("../src/Orthogonalization.jl")
include("../src/Arnoldi.jl")
include("../src/ImplicitRestart.jl")
include("../src/Eigenpairs.jl")
include("../src/BadRestart.jl")

## Validation

We start by randomly picking matrices from the SuiteSparse Matrix Collection.

We validate it on $3$ matries of size $2000 \times 2000$, while trying to find the $6$ largest eigenpairs.
We pick the matrices from "gravity", "chebspec" and "lotkin".
We compare the maximum relative error between the eigenvalues, and the maximum accuracy given by $\|Av - \lambda v\|$ where $(\lambda, v)$ is the computed eigenpair with $v$ normalized.


In [ ]:
n = 2000
num_eig = 6
A1 = MatrixDepot.gravity(n);
A2 = MatrixDepot.chebspec(n);
A3 = MatrixDepot.lotkin(n);

In [ ]:
function accuracy_eigvals(real_evals, approx_evals)
    cost_matrix = [abs.((real_evals[i] - approx_evals[j])/real_evals[i]) for i in 1:length(real_evals), j in 1:length(approx_evals)]
    assignment, cost = hungarian(cost_matrix)
    return cost
end

### Gravity Matrix

In [ ]:
s_grid_A1 = Int.(range(7, 20, 14))

real_evals_A1, real_evecs_A1 = eigen(A1, sortby=x -> (-abs(x), real(x), imag(x)))
real_evecs_A1 = real_evecs_A1[:, 1:num_eig]
real_evals_A1 = real_evals_A1[1:num_eig]

println("Real eigenvalues: ", abs.(real_evals_A1))

In [ ]:
evals = eigvals(A1)

p = scatter(real(evals), imag(evals), color=:blue,
     title = "Eigenvalues of Gravity matrix", xlabel="Real Part", ylabel="Imaginary Part",
     xlim=[-1, 7], ylim = [-1, 1], legend=:top,
     markerstrokewidth=0, markersize=3, label="")

savefig(p, "../fig/Validation/evals_gravity.png")

In [ ]:
error_arnoldi_A1 = []
error_naive_restart_A1 = []
error_iram_A1 = []

accuracy_eigvals_arnoldi_A1 = []
accuracy_eigvals_naive_restart_A1 = []
accuracy_eigvals_iram_A1 = []

@showprogress for s in s_grid_A1
    approx_evals, approx_evecs = Eigenpairs.eigenpairs_arnoldi(A1, n, num_eig=num_eig, subspace_dim=s, tol=0.0)
    error = maximum(abs.(A1 * approx_evecs - approx_evecs * Diagonal(approx_evals)))
    accuracy = accuracy_eigvals(real_evals_A1, approx_evals)
    push!(accuracy_eigvals_arnoldi_A1, accuracy)
    push!(error_arnoldi_A1, error)
end

@showprogress for s in s_grid_A1
    approx_evals, approx_evecs, _ = Eigenpairs.eigenpairs_naive_restart(A1, n, num_eig=num_eig, max_iter=50, subspace_dim=s, restart_dim=max(floor(Int, s/2), num_eig), tol=0.0)
    error = maximum(abs.(A1 * approx_evecs - approx_evecs * Diagonal(approx_evals)))
    accuracy = accuracy_eigvals(real_evals_A1, approx_evals)
    push!(accuracy_eigvals_naive_restart_A1, accuracy)
    push!(error_naive_restart_A1, error)
end

@showprogress for s in s_grid_A1
    approx_evals, approx_evecs, _ = Eigenpairs.eigenpairs_iram(A1, n, num_eig=num_eig, max_iter=50, subspace_dim=s, restart_dim=max(floor(Int, s/2), num_eig), tol=0.0)
    error = maximum(abs.(A1 * approx_evecs - approx_evecs * Diagonal(approx_evals)))
    accuracy = accuracy_eigvals(real_evals_A1, approx_evals)
    push!(accuracy_eigvals_iram_A1, accuracy)
    push!(error_iram_A1, error)
end

println("Errors for Arnoldi: ", error_arnoldi_A1)
println("Errors for Naive Restart: ", error_naive_restart_A1)
println("Errors for IRAM: ", error_iram_A1)

println("Accuracy for Arnoldi: ", accuracy_eigvals_arnoldi_A1)
println("Accuracy for Naive Restart: ", accuracy_eigvals_naive_restart_A1)
println("Accuracy for IRAM: ", accuracy_eigvals_iram_A1)

In [ ]:
p = plot(s_grid_A1, error_arnoldi_A1, color=:blue,
     title = L"Residual $\max_i \, |\!|Av_i - \lambda_i v_i|\!|_1$", xlabel="Krylov subspace dimension", ylabel="Residual",
     yaxis=:log,
     xlim=[7, 20], ylim = [1e-18, 1], legend=:top,
     label = "Arnoldi")

plot!(p, s_grid_A1, error_naive_restart_A1, color=:green,
     label = "Naive Restart")

plot!(p, s_grid_A1, error_iram_A1, color=:red,
     label = "IRAM")

savefig(p, "../fig/Validation/error_gravity.png")

In [ ]:
p = plot(s_grid_A1, accuracy_eigvals_arnoldi_A1, color=:blue,
     title = L"Accuracy $\max_i \frac{|\lambda_i - \hat{\lambda}_i|}{|\lambda_i|}$", xlabel="Krylov subspace dimension", ylabel="Accuracy",
     yaxis=:log,
     xlim=[7, 20], ylim = [1e-18, 1], legend=:top,
     label = "Arnoldi")

plot!(p, s_grid_A1, accuracy_eigvals_naive_restart_A1, color=:green,
     label = "Naive Restart")

plot!(p, s_grid_A1, accuracy_eigvals_iram_A1, color=:red,
     label = "IRAM")

savefig(p, "../fig/Validation/accuracy_gravity.png")

In [ ]:
maximum_iterations = 50

iter_grid = Int.(range(1, maximum_iterations, maximum_iterations))

residuals_iram_A1 = zeros(length(iter_grid), length(s_grid_A1))

it = 1

@showprogress for s in s_grid_A1[1:5]
    _, _, residual_iram = Eigenpairs.eigenpairs_iram(A1, n, num_eig=num_eig, max_iter=maximum_iterations, subspace_dim=s, restart_dim=max(floor(Int, s/2), num_eig), tol=0.0);
    residuals_iram_A1[:, it] = residual_iram
    it += 1
end

In [ ]:
my_labels = reshape(["dimension $(s_grid_A1[i])" for i in 1:5], 1, :)

p = plot(residuals_iram_A1[:, 1:5], label = my_labels,
        title = L"Residual $\max_i \, |\!|Av_i - \lambda_i v_i|\!|_1$", xlabel = "Restart iterations", ylabel="Residual",
        yaxis=:log, xlim=[0, 50], ylim = [1e-18, 1], legend=:topright)

savefig(p, "../fig/Validation/iram_gravity.png")

### Chebyshev Spectral Differentiation Matrix

In [ ]:
s_grid_A2 = Int.(range(10, 60, 26))

real_evals_A2, real_evecs_A2 = eigen(A2, sortby= x -> (-abs(x), real(x), imag(x)))
real_evecs_A2 = real_evecs_A2[:, 1:num_eig]
real_evals_A2 = real_evals_A2[1:num_eig]

println("Real eigenvalues: ", real_evals_A2)

In [ ]:
evals = eigvals(A2)

p = scatter(real(evals), imag(evals), color=:blue,
     title = "Eigenvalues of Chebspec matrix", xlabel="Real Part", ylabel="Imaginary Part",
     xlim=[-90000, 90000], ylim = [-45000, 45000], legend=:top,
     markerstrokewidth=0, markersize=3, label="")

savefig(p, "../fig/Validation/evals_chebspec.png")

In [ ]:
error_arnoldi_A2 = []
error_naive_restart_A2 = []
error_iram_A2 = []

accuracy_eigvals_arnoldi_A2 = []
accuracy_eigvals_naive_restart_A2 = []
accuracy_eigvals_iram_A2 = []

@showprogress for s in s_grid_A2
    approx_evals, approx_evecs = Eigenpairs.eigenpairs_arnoldi(A2, n, num_eig=num_eig, subspace_dim=s, tol=0.0)
    error = maximum(abs.(A2 * approx_evecs - approx_evecs * Diagonal(approx_evals)))
    accuracy = accuracy_eigvals(real_evals_A2, approx_evals)
    push!(accuracy_eigvals_arnoldi_A2, accuracy)
    push!(error_arnoldi_A2, error)
end

@showprogress for s in s_grid_A2
    approx_evals, approx_evecs, _ = Eigenpairs.eigenpairs_naive_restart(A2, n, num_eig=num_eig, max_iter=50, subspace_dim=s, restart_dim=max(floor(Int, s/2), num_eig), tol=0.0)
    error = maximum(abs.(A2 * approx_evecs - approx_evecs * Diagonal(approx_evals)))
    accuracy = accuracy_eigvals(real_evals_A2, approx_evals)
    push!(accuracy_eigvals_naive_restart_A2, accuracy)
    push!(error_naive_restart_A2, error)
end

@showprogress for s in s_grid_A2
    approx_evals, approx_evecs, _ = Eigenpairs.eigenpairs_iram(A2, n, num_eig=num_eig, max_iter=50, subspace_dim=s, restart_dim=max(floor(Int, s/2), num_eig), tol=0.0)
    error = maximum(abs.(A2 * approx_evecs - approx_evecs * Diagonal(approx_evals)))
    accuracy = accuracy_eigvals(real_evals_A2, approx_evals)
    push!(accuracy_eigvals_iram_A2, accuracy)
    push!(error_iram_A2, error)
end

println("Errors for Arnoldi: ", error_arnoldi_A2)
println("Errors for Naive Restart: ", error_naive_restart_A2)
println("Errors for IRAM: ", error_iram_A2)

println("Accuracy for Arnoldi: ", accuracy_eigvals_arnoldi_A2)
println("Accuracy for Naive Restart: ", accuracy_eigvals_naive_restart_A2)
println("Accuracy for IRAM: ", accuracy_eigvals_iram_A2)

In [ ]:
p = plot(s_grid_A2, error_arnoldi_A2, color=:blue,
     title = L"Residual $\max_i \, |\!|Av_i - \lambda_i v_i|\!|_1$", xlabel="Krylov subspace dimension", ylabel="Residual",
     yaxis=:log,
     xlim=[10, 60], ylim = [1e-10, 1e4], legend=:top,
     label = "Arnoldi")

plot!(p, s_grid_A2, error_naive_restart_A2, color=:green,
     label = "Naive Restart")

plot!(p, s_grid_A2, error_iram_A2, color=:red,
     label = "IRAM")

savefig(p, "../fig/Validation/error_chebspec.png")

In [ ]:
p = plot(s_grid_A2, accuracy_eigvals_arnoldi_A2, color=:blue,
     title = L"Accuracy $\max_i \frac{|\lambda_i - \hat{\lambda}_i|}{|\lambda_i|}$", xlabel="Krylov subspace dimension", ylabel="Accuracy",
     yaxis=:log,
     xlim=[10, 60], ylim = [1e-10, 1e4], legend=:top,
     label = "Arnoldi")

plot!(p, s_grid_A2, accuracy_eigvals_naive_restart_A2, color=:green,
     label = "Naive Restart")

plot!(p, s_grid_A2, accuracy_eigvals_iram_A2, color=:red,
     label = "IRAM")

savefig(p, "../fig/Validation/accuracy_chebspec.png")

In [ ]:
maximum_iterations = 50

iter_grid = Int.(range(1, maximum_iterations, maximum_iterations))

residuals_iram_A2 = zeros(length(iter_grid), length(s_grid_A2))

it = 1

@showprogress for s in s_grid_A2[1:10]
    _, _, residual_iram = Eigenpairs.eigenpairs_iram(A2, n, num_eig=num_eig, max_iter=maximum_iterations, subspace_dim=s, restart_dim=max(floor(Int, s/2), num_eig), tol=0.0);
    residuals_iram_A2[:, it] = residual_iram
    it += 1
end

In [ ]:
my_labels = reshape(["dimension $(s_grid_A2[i])" for i in 1:3:10], 1, :)


p = plot(residuals_iram_A2[:, 1:3:10], label = my_labels,
        title = L"Residual $\max_i \, |\!|Av_i - \lambda_i v_i|\!|_1$", xlabel = "Restart iterations", ylabel="Residual",
        yaxis=:log, xlim=[0, 50], ylim = [1e-12, 1e15], legend=:topright)

savefig(p, "../fig/Validation/iram_chebspec.png")

### Lotkin Matrix

In [ ]:
s_grid_A3 = Int.(range(7, 20, 14))

real_evals_A3, real_evecs_A3 = eigen(A3, sortby= x -> -abs(x))
real_evecs_A3 = real_evecs_A3[:, 1:num_eig]
real_evals_A3 = real_evals_A3[1:num_eig]

println("Real eigenvalues: ", abs.(real_evals_A3))

In [ ]:
evals = eigvals(A3)

p = scatter(real(evals), imag(evals), color=:blue,
     title = "Eigenvalues of Lotkin matrix", xlabel="Real Part", ylabel="Imaginary Part",
     xlim=[-0.1, 5], ylim = [-1, 1], legend=:top,
     markerstrokewidth=0, markersize=3, label="")

savefig(p, "../fig/Validation/evals_lotkin.png")

In [ ]:
error_arnoldi_A3 = []
error_naive_restart_A3 = []
error_iram_A3 = []

accuracy_eigvals_arnoldi_A3 = []
accuracy_eigvals_naive_restart_A3 = []
accuracy_eigvals_iram_A3 = []

@showprogress for s in s_grid_A3
    approx_evals, approx_evecs = Eigenpairs.eigenpairs_arnoldi(A3, n, num_eig=num_eig, subspace_dim=s, tol=0.0)
    error = maximum(abs.(A3 * approx_evecs - approx_evecs * Diagonal(approx_evals)))
    accuracy = accuracy_eigvals(real_evals_A3, approx_evals)
    push!(accuracy_eigvals_arnoldi_A3, accuracy)
    push!(error_arnoldi_A3, error)
end

@showprogress for s in s_grid_A3
    approx_evals, approx_evecs, _ = Eigenpairs.eigenpairs_naive_restart(A3, n, num_eig=num_eig, max_iter=50, subspace_dim=s, restart_dim=max(floor(Int, s/2), num_eig), tol=0.0)
    error = maximum(abs.(A3 * approx_evecs - approx_evecs * Diagonal(approx_evals)))
    accuracy = accuracy_eigvals(real_evals_A3, approx_evals)
    push!(accuracy_eigvals_naive_restart_A3, accuracy)
    push!(error_naive_restart_A3, error)
end

@showprogress for s in s_grid_A3
    approx_evals, approx_evecs, _ = Eigenpairs.eigenpairs_iram(A3, n, num_eig=num_eig, max_iter=50, subspace_dim=s, restart_dim=max(floor(Int, s/2), num_eig), tol=0.0)
    error = maximum(abs.(A3 * approx_evecs - approx_evecs * Diagonal(approx_evals)))
    accuracy = accuracy_eigvals(real_evals_A3, approx_evals)
    push!(accuracy_eigvals_iram_A3, accuracy)
    push!(error_iram_A3, error)
end

println("Errors for Arnoldi: ", error_arnoldi_A3)
println("Errors for Naive Restart: ", error_naive_restart_A3)
println("Errors for IRAM: ", error_iram_A3)

println("Accuracy for Arnoldi: ", accuracy_eigvals_arnoldi_A3)
println("Accuracy for Naive Restart: ", accuracy_eigvals_naive_restart_A3)
println("Accuracy for IRAM: ", accuracy_eigvals_iram_A3)

In [ ]:
p = plot(s_grid_A3, error_arnoldi_A3, color=:blue,
     title = L"Residual $\max_i \, |\!|Av_i - \lambda_i v_i|\!|_1$", xlabel="Krylov subspace dimension", ylabel="Residual",
     yaxis=:log,
     xlim=[7, 20], ylim = [1e-18, 1], legend=:top,
     label = "Arnoldi")

plot!(p, s_grid_A3, error_naive_restart_A3, color=:green,
     label = "Naive Restart")

plot!(p, s_grid_A3, error_iram_A3, color=:red,
     label = "IRAM")

savefig(p, "../fig/Validation/error_lotkin.png")

In [ ]:
p = plot(s_grid_A3, accuracy_eigvals_arnoldi_A3, color=:blue,
     title = L"Accuracy $\max_i \frac{|\lambda_i - \hat{\lambda}_i|}{|\lambda_i|}$", xlabel="Krylov subspace dimension", ylabel="Accuracy",
     yaxis=:log,
     xlim=[7, 20], ylim = [1e-18, 1e5], legend=:top,
     label = "Arnoldi")

plot!(p, s_grid_A3, accuracy_eigvals_naive_restart_A3, color=:green,
     label = "Naive Restart")

plot!(p, s_grid_A3, accuracy_eigvals_iram_A3, color=:red,
     label = "IRAM")

savefig(p, "../fig/Validation/accuracy_lotkin.png")

In [ ]:
maximum_iterations = 50

iter_grid = Int.(range(1, maximum_iterations, maximum_iterations))

residuals_iram_A3 = zeros(length(iter_grid), length(s_grid_A3))

it = 1

@showprogress for s in s_grid_A3[1:6]
    _, _, residual_iram = Eigenpairs.eigenpairs_iram(A3, n, num_eig=num_eig, max_iter=maximum_iterations, subspace_dim=s, restart_dim=max(floor(Int, s/2), num_eig), tol=0.0);
    residuals_iram_A3[:, it] = residual_iram
    it += 1
end

In [ ]:
my_labels = reshape(["dimension $(s_grid_A3[i])" for i in 1:6], 1, :)

p = plot(residuals_iram_A3[:, 1:6], label = my_labels,
        title = L"Residual $\max_i \, |\!|Av_i - \lambda_i v_i|\!|_1$", xlabel = "Restart iterations", ylabel="Residual",
        yaxis=:log, xlim=[0, 50], ylim = [1e-18, 1e6], legend=:topright)

savefig(p, "../fig/Validation/iram_lotkin.png")